In [ ]:
import json
import sys
import numpy as np
import copy

sys.path.append('../')

from pqcqec.noise.builder import apply_gate_sequence_noise
# from pqcqec.noise.simple_noise import PennylaneNoisyGates
from pqcqec.simulate.simulate import get_input_data 
from pqcqec.simulate.jax_statevector import build_jax_circuit, jax_run_many_states
from pqcqec.training.jax_loss_functions import jax_pure_state_fidelity_batched

In [2]:
# DATA_PATH = '../' + 'nogit/sequence_noise/rzrxrz/1q_10g_10blk_data/'# 
DATA_PATH = '../' + 'nogit/sequence_noise/'
SEQUENCE_FINE_TUNED_DATA_FILE = DATA_PATH + 'gt_fid2.jsonl'

with open(SEQUENCE_FINE_TUNED_DATA_FILE, 'r') as f:
    lines = f.readlines()
    data = [json.loads(line) for line in lines]
    # data = json.load(f)

print(f"Total data points loaded: {len(data)}")
print(f"Example data point: {data[0]}")

Total data points loaded: 1001
Example data point: {'index': 0, 'fidelity': 1.0, 'angles': [1.5974044799804688e-05, 0.0, 0.0], 'base_len': 100, 'base_tokens': ['z', 'z', 'h', 'x', 'h', 'h', 'z', 'x', 'h', 'h', 'z', 'h', 'x', 'z', 'h', 'x', 'z', 'z', 'z', 'z', 'x', 'z', 'z', 'z', 'h', 'x', 'h', 'h', 'z', 'z', 'h', 'z', 'x', 'z', 'h', 'x', 'z', 'h', 'z', 'z', 'x', 'h', 'z', 'x', 'x', 'z', 'x', 'h', 'x', 'z', 'z', 'h', 'x', 'x', 'h', 'z', 'x', 'h', 'z', 'h', 'z', 'h', 'z', 'h', 'h', 'h', 'h', 'h', 'h', 'x', 'x', 'x', 'h', 'h', 'h', 'x', 'z', 'z', 'z', 'z', 'z', 'z', 'h', 'h', 'z', 'x', 'z', 'z', 'z', 'h', 'z', 'h', 'h', 'h', 'z', 'z', 'z', 'x', 'h', 'h'], 'K': 1000, 'baseline': 'clean', 'noise': {'mode': 'mutation', 'rules': 'HH->HX,XX->XZ,ZZ->ZH'}, 'mutated_tokens': ['z', 'h', 'h', 'x', 'h', 'x', 'z', 'x', 'h', 'x', 'z', 'h', 'x', 'z', 'h', 'x', 'z', 'h', 'z', 'h', 'x', 'z', 'h', 'z', 'h', 'x', 'h', 'x', 'z', 'h', 'h', 'z', 'x', 'z', 'h', 'x', 'z', 'h', 'z', 'h', 'x', 'h', 'z', 'x', 'z',

In [3]:
NUM_QUBITS = 1
NUM_VALS = 1000

In [4]:
poor_mutated_fid_data = []
poor_fidelity_noise_data = []
for i, entry in enumerate(data):
    base_ops = [(gate, [0], []) for gate in entry['base_tokens']]
    mutated_ops = [(gate, [0], []) for gate in entry['mutated_tokens']]
    # base_ops = entry['base_circuit_tokens']
    # mutated_ops = entry['pqc_circuit_tokens']
    
    noisy_ops = apply_gate_sequence_noise(
        base_ops,
        noise={
            ('h', 'h'): ('h', 'x'),   # HH → HX
            ('x', 'x'): ('x', 'z'),   # XX → XZ
            ('z', 'z'): ('z', 'h'),   # ZZ → ZH
        }
    )

    # noise_model = PennylaneNoisyGates()

    # x_noise_arr = np.random.uniform(noise_model.x_noise_min, noise_model.x_noise_max, 
    #                                 (len(base_ops),)).astype(np.float32)
    # z_noise_arr = np.random.uniform(noise_model.z_noise_min, noise_model.z_noise_max, 
    #                                 (len(base_ops),)).astype(np.float32)
    
    # noisy_test_ops = []
    # for i, op in enumerate(noisy_ops):
    #     noisy_test_ops.append(op)
    #     gate, qubits, params = op
    #     for q in qubits:
    #         noisy_test_ops.append(('rx', [q], [float(x_noise_arr[min(i, len(x_noise_arr)-1)])]))
    #         noisy_test_ops.append(('rz', [q], [float(z_noise_arr[min(i, len(z_noise_arr)-1)])]))
    
    pqc_params = entry['angles']
    # pqc_params = entry['pqc_params']
    noisy_ops.append(('rz', [0], [pqc_params[0]]))
    noisy_ops.append(('rx', [0], [pqc_params[1]]))
    noisy_ops.append(('rz', [0], [pqc_params[2]]))

    mutated_ops.append(('rz', [0], [pqc_params[0]]))
    mutated_ops.append(('rx', [0], [pqc_params[1]]))
    mutated_ops.append(('rz', [0], [pqc_params[2]]))

    input_data = get_input_data(NUM_QUBITS, NUM_VALS, i)

    base_circuit = build_jax_circuit(base_ops)
    ideal_measured_states = jax_run_many_states(
        NUM_QUBITS, *base_circuit, input_data
    )

    noisy_circuit = build_jax_circuit(noisy_ops)
    noisy_measured_states = jax_run_many_states(
        NUM_QUBITS, *noisy_circuit, input_data
    )

    mutated_circuit = build_jax_circuit(mutated_ops)
    mutated_measured_states = jax_run_many_states(
        NUM_QUBITS, *mutated_circuit, input_data
    )

    noisy_fid = jax_pure_state_fidelity_batched(
        ideal_measured_states, noisy_measured_states
    )

    mutated_fid = jax_pure_state_fidelity_batched(
        ideal_measured_states, mutated_measured_states
    )

    if np.mean(noisy_fid) > 0.99 and np.mean(mutated_fid) < 0.99:
        new_entry = copy.deepcopy(entry)
        new_entry['noisy_tokens'] = [op[0] for op in noisy_ops]
        poor_mutated_fid_data.append(new_entry)
        # print(f"Entry {i} has poor fidelity: {np.mean(pqc_fid)}\n\t Base ops: {entry['base_tokens']}\n\tNoisy ops: {entry['mutated_tokens']}\n\t PQC angles: {pqc_params}")

    elif np.mean(noisy_fid) < 0.99 and np.mean(mutated_fid) < 0.99:
        new_entry = copy.deepcopy(entry)
        new_entry['noisy_tokens'] = [op[0] for op in noisy_ops]
        poor_fidelity_noise_data.append(new_entry)
        # print(f"Entry {i} has poor fidelity: {np.mean(pqc_fid)}\n\t Base ops: {entry['base_tokens']}\n\tNoisy ops: {entry['mutated_tokens']}\n\t PQC angles: {pqc_params}")

    elif np.mean(noisy_fid) < 0.99 and np.mean(mutated_fid) > 0.99:
        print(f"Entry {i} has poor Noisy Fid {np.mean(noisy_fid)}, Good Mutated Fid {np.mean(mutated_fid)}")












In [5]:
print(f"Total entries with poor mutated fidelity and good noisy fidelity: {len(poor_mutated_fid_data)}")
print(f"Total entries with poor fidelity noise and poor mutated fidelity: {len(poor_fidelity_noise_data)}")

Total entries with poor mutated fidelity and good noisy fidelity: 759
Total entries with poor fidelity noise and poor mutated fidelity: 0


In [6]:
for entry in poor_mutated_fid_data:
    print(f" Base ops:\t{entry['base_tokens']}\n Noisy ops:\t{entry['noisy_tokens']}\n Mutat ops:\t{entry['mutated_tokens']}\n")

 Base ops:	['x', 'z', 'z', 'x', 'h', 'h', 'h', 'z', 'x', 'x', 'z', 'h', 'z', 'x', 'h', 'z', 'x', 'z', 'z', 'x', 'x', 'h', 'z', 'h', 'x', 'h', 'x', 'x', 'h', 'h', 'x', 'x', 'x', 'h', 'x', 'x', 'z', 'h', 'h', 'x', 'z', 'z', 'x', 'x', 'z', 'z', 'z', 'h', 'z', 'z', 'x', 'h', 'z', 'z', 'h', 'h', 'x', 'x', 'z', 'h', 'x', 'h', 'z', 'z', 'h', 'h', 'h', 'z', 'h', 'h', 'h', 'x', 'x', 'z', 'z', 'h', 'h', 'x', 'h', 'z', 'z', 'h', 'z', 'x', 'h', 'z', 'h', 'h', 'x', 'h', 'z', 'x', 'z', 'z', 'z', 'z', 'z', 'h', 'h', 'h']
 Noisy ops:	['x', 'z', 'h', 'x', 'h', 'x', 'x', 'z', 'x', 'z', 'z', 'h', 'z', 'x', 'h', 'z', 'x', 'z', 'h', 'x', 'z', 'h', 'z', 'h', 'x', 'h', 'x', 'z', 'h', 'x', 'x', 'z', 'z', 'h', 'x', 'z', 'z', 'h', 'x', 'x', 'z', 'h', 'x', 'z', 'z', 'h', 'h', 'h', 'z', 'h', 'x', 'h', 'z', 'h', 'h', 'x', 'x', 'z', 'z', 'h', 'x', 'h', 'z', 'h', 'h', 'x', 'x', 'z', 'h', 'x', 'x', 'x', 'z', 'z', 'h', 'h', 'x', 'x', 'h', 'z', 'h', 'h', 'z', 'x', 'h', 'z', 'h', 'x', 'x', 'h', 'z', 'x', 'z', 'h', 'h', 